# Tool Function

## Steps

1. Write a Tool function
2. Write a JSON Schema
3. Call LLM with JSON Schema
4. Run Tool
5. Add Tool result and call Claude again

In [144]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

### Create Tool Function (Get Current Date & Time) with JSON Schema

In [147]:
from anthropic.types import ToolParam

def get_current_datetime(dateformat="%Y-%m-%d %H:%M:%S"):
    # Input validation
    if not dateformat:
        raise ValueError("Date format string cannot be empty.")
    return datetime.now().strftime(dateformat)

# Default format: "2026-03-29 14:30:25"
# get_current_datetime()

# Just hour and minute: "14:30"
# get_current_datetime("%H:%M")

get_current_datetime_schema = ToolParam({
  "name": "get_current_datetime",
  "description": "Returns the current date and/or time formatted as a string. Use this tool when the user asks what time or date it is, or needs a timestamp in a specific format. The format string uses Python's strftime directives (e.g., '%Y-%m-%d', '%H:%M:%S', '%A %B %d, %Y').",
  "input_schema": {
    "type": "object",
    "properties": {
      "dateformat": {
        "type": "string",
        "description": "A Python strftime-compatible format string that controls the structure of the returned datetime string. Defaults to '%Y-%m-%d %H:%M:%S' (e.g., '2025-03-29 14:30:00'). Must be a non-empty string. Common examples: '%Y-%m-%d' for date only, '%H:%M:%S' for time only, '%A %B %d, %Y' for a human-readable date.",
        "default": "%Y-%m-%d %H:%M:%S"
      }
    },
    "required": []
  }
})

> Note: Using `ToolParam` to enclose the JSON Schema is not required, but added to avoid Type Errors (if it happens) --> Keep the code more Robust.

### Call LLM with JSON Schema

In [ ]:
# Mocking a call to the tool in a prompt:

messages = []

messages.append({
    "role": "user",
    "content": "What is the exact time now? Tell me in Hour:Minute format."
})

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

response

Message(id='msg_01CKwiqJUVz9Lu3bNz5FJDHY', container=None, content=[ToolUseBlock(id='toolu_01QQFqxdeqRjoKFDGZoEJsBj', caller=DirectCaller(type='direct'), input={'dateformat': '%H:%M'}, name='get_current_datetime', type='tool_use')], model='claude-haiku-4-5-20251001', role='assistant', stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=767, output_tokens=60, server_tool_use=None, service_tier='standard'))

> Note: Note that the LLM has requested use of a tool using two responses: `TextBlock`_(Let me find that info...)_ and `ToolUseBlock` (as follows)
    
```code
...ToolUseBlock(
    ...
    caller=DirectCaller(type='direct'),
    input={'dateformat': '%Y-%m-%d %H:%M:%S'},
    name='get_current_datetime',
    type='tool_use')...
... role='assistant', stop_reason='tool_use'...
```

**Add the assistant response to the messages so that the tool call is included in the conversation history, which allows you to then trigger the tool function based on the LLM's response content.**

In [149]:
messages.append({
    "role": "assistant",
    "content": response.content
})

messages

[{'role': 'user',
  'content': 'What is the exact time now? Tell me in Hour:Minute format.'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01QQFqxdeqRjoKFDGZoEJsBj', caller=DirectCaller(type='direct'), input={'dateformat': '%H:%M'}, name='get_current_datetime', type='tool_use')]}]

**Extract the exact input sent by LLM**

In [150]:
response.content[0].input 
# LLM is passing this input to the tool based on the user's question 
# Use this to trigger the tool function and get the output, then add that output back to the messages and continue the conversation!

{'dateformat': '%H:%M'}

### Run Tool

In [ ]:
result = get_current_datetime(response.content[0].input["dateformat"]) # from the tool function
# Will return current datetime in the user-requested format 

result

'05:25'

### Pass Tool result back to LLM

**Add the tool result contents to the user message**

- But why a "user" message? Because the tool is not created by the assistant, but by the client application run by the user. The LLM simply calls it and the client responds with the tool result.
- Look at the complex structure of `content` here, unlike a simple text - we are passing a `tool_result` response to LLM.
- The `tool_result` response block contains `type`, `tool_use_id`, `content`, `is_error`

In [152]:
messages.append({
    "role": "user",
    "content": [{
        "type": "tool_result",
        "tool_use_id": response.content[0].id,
        "content": result,
        "is_error": False
    }]
})

messages

[{'role': 'user',
  'content': 'What is the exact time now? Tell me in Hour:Minute format.'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01QQFqxdeqRjoKFDGZoEJsBj', caller=DirectCaller(type='direct'), input={'dateformat': '%H:%M'}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01QQFqxdeqRjoKFDGZoEJsBj',
    'content': '05:25',
    'is_error': False}]}]

**Pass in the appended message block with full conversation history (request, tool use, tool result, etc.) to get the final response**

In [154]:
response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema] # Need to include the tool schema here as well in case the model needs to call it again based on the conversation
)

response.content[0].text

'The exact time now is **05:25** (5:25 AM).'